# Extraction evaluation (not production)

Offline checks for stage 2: field precision/recall, completeness, evidence keep rate, and evidence *support* labels (substring match is not support).

| Sibling file | Role |
|---|---|
| `extraction_gold.json` | 8 thesis-related papers with method/dataset/metric labels |
| `extraction_scoring.py` | Precision/recall helpers |
| `extraction_eval.json` | Optional saved live run |

Default `RUN_LIVE = False`: score a saved `extraction_eval.json` if present; otherwise only show gold.

Set `RUN_LIVE = True` to extract gold papers (needs network + an LLM key). Ablation compares abstract-only vs selected-sections by changing `SectionPolicy` only.

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

from research_assistant.config import ExtractionConfig, REPO_ROOT

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from evaluation.extraction_scoring import (
    completeness,
    evidence_keep_rate,
    evidence_precision,
    mention_values,
    precision_recall,
)

EVAL_DIR = REPO_ROOT / "evaluation"
GOLD_PATH = EVAL_DIR / "extraction_gold.json"
SAVED_PATH = EVAL_DIR / "extraction_eval.json"

RUN_LIVE = True
ABLATION = True  # when RUN_LIVE, also extract with abstract-only selector

gold = json.loads(GOLD_PATH.read_text(encoding="utf-8"))
print("EVAL_DIR:", EVAL_DIR)
print("Gold papers:", len(gold["papers"]))
print("RUN_LIVE:", RUN_LIVE)

EVAL_DIR: /home/liu/Code/Projects/Research-Assistance-Agent/evaluation
Gold papers: 8
RUN_LIVE: True


## Score a saved extraction run


In [2]:
def extraction_records(payload) -> list[dict]:
    """Unwrap ExtractionResult JSON or an eval payload with selected/abstract_only."""
    if isinstance(payload, list):
        return payload
    if not isinstance(payload, dict):
        return []
    records = payload.get("records")
    if isinstance(records, list):
        return records
    selected = payload.get("selected")
    if isinstance(selected, dict) and isinstance(selected.get("records"), list):
        return selected["records"]
    if isinstance(selected, list):
        return selected
    return []


def score_records(records: list[dict], gold_papers: list[dict]) -> dict:
    by_id = {p["arxiv_id"]: p for p in gold_papers}
    field_rows = []
    support_labels = []
    for rec in records:
        g = by_id.get(rec.get("arxiv_id"))
        if not g:
            continue
        for field, gold_key in (
            ("method_keywords", "method_keywords"),
            ("datasets", "datasets"),
            ("metrics", "metrics"),
        ):
            stats = precision_recall(mention_values(rec, field), g.get(gold_key) or [])
            field_rows.append({"arxiv_id": rec["arxiv_id"], "field": field, **stats})
        for sample in g.get("evidence_samples") or []:
            support_labels.append(sample.get("support", "unsupported"))
    return {
        "completeness": completeness(records),
        "evidence_keep_rate": evidence_keep_rate(records),
        "evidence_precision": evidence_precision(support_labels),
        "fields": field_rows,
        "n_scored": sum(1 for r in records if r.get("arxiv_id") in by_id),
    }


if SAVED_PATH.is_file():
    saved = json.loads(SAVED_PATH.read_text(encoding="utf-8"))
    selected_records = extraction_records(saved)
    print("Selected:")
    print(json.dumps(score_records(selected_records, gold["papers"]), indent=2)[:2000])
    abstract_payload = saved.get("abstract_only") if isinstance(saved, dict) else None
    if isinstance(abstract_payload, dict):
        print("\nAbstract-only:")
        print(json.dumps(score_records(extraction_records(abstract_payload), gold["papers"]), indent=2)[:2000])
else:
    print("No extraction_eval.json yet. Run with RUN_LIVE=True or:")
    print(
        "python -m research_assistant.extraction "
        "--arxiv-id 2205.09329 --json-out evaluation/extraction_eval.json"
    )

No extraction_eval.json yet. Run with RUN_LIVE=True or:
python -m research_assistant.extraction --arxiv-id 2205.09329 --json-out evaluation/extraction_eval.json


## Live extraction + selector ablation


In [3]:
def eval_config(**kwargs) -> ExtractionConfig:
    defaults = dict(
        llm_concurrency=1,
        max_retries=0,
        llm_batch_size=5,
        max_llm_calls_per_run=8,
        llm_min_interval_s=5.0,
        skip_llm_if_confident=False,
    )
    defaults.update(kwargs)
    return ExtractionConfig(**defaults)


if RUN_LIVE:
    from research_assistant.extraction import extract_papers
    from research_assistant.retrieval.types import PaperHit

    papers = [
        PaperHit(
            rank=i + 1,
            row_id=-1,
            arxiv_id=p["arxiv_id"],
            title=p["title"],
            latest_version="v1",
        )
        for i, p in enumerate(gold["papers"])
    ]
    selected = extract_papers(papers, eval_config())
    payload = {"selected": json.loads(selected.model_dump_json())}
    print("Selected metrics:", selected.metrics)
    print("Selected score:", score_records(payload["selected"]["records"], gold["papers"]))
    if ABLATION:
        abstract = extract_papers(papers, eval_config(abstract_only=True))
        payload["abstract_only"] = json.loads(abstract.model_dump_json())
        print("Abstract-only metrics:", abstract.metrics)
        print(
            "Abstract-only score:",
            score_records(payload["abstract_only"]["records"], gold["papers"]),
        )
    SAVED_PATH.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    print("Wrote", SAVED_PATH)
else:
    print("RUN_LIVE is False; not calling arXiv / LLM.")

Selected metrics: {'n_papers': 8, 'ok': 8, 'degraded': 0, 'skipped': 0, 'tex_rate': 1.0, 'pdf_fallback_rate': 0.0, 'abstract_fallback_rate': 0.0, 'skip_rate': 0.0, 'evidence_keep_rate': 1.0, 'section_relocate_rate': 0.0, 'cache_hits': 0, 'mean_selected_chars': 23956.2, 'mean_candidate_chars': 4373.4, 'seconds': 47.712, 'llm_calls': 2, 'llm_skipped_confident': 0}
Selected score: {'completeness': 1.0, 'evidence_keep_rate': 1.0, 'evidence_precision': {'supported': 0.667, 'partially_supported': 0.0, 'unsupported': 0.333, 'n': 3}, 'fields': [{'arxiv_id': '2205.09329', 'field': 'method_keywords', 'precision': 1.0, 'recall': 0.333, 'f1': 0.5, 'jaccard': 0.333}, {'arxiv_id': '2205.09329', 'field': 'datasets', 'precision': 1.0, 'recall': 0.667, 'f1': 0.8, 'jaccard': 0.667}, {'arxiv_id': '2205.09329', 'field': 'metrics', 'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'jaccard': 1.0}, {'arxiv_id': '2107.07075', 'field': 'method_keywords', 'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'jaccard': 0.0}, 